In [ ]:
import random, math
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import Aer, AerSimulator
from qiskit import transpile

## BB84
The first protocol we will investigate is the BB84:
1. Alice generates random bits $(X_1, X_2, \dots X_n)$ and shares with Bob a sequence of qubits prepared in one of the four states $\{\ket{0}, \ket{1}, \ket{+}, \ket{-}\}$, eg
$$ \ket{1}\ket{+}\ket{+}\ket{0}\ket{-}\ket{0}\ket{-}\ket{0}\ket{1}\ket{+}\ket{1}\ket{0}$$

| qubit   | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---------|---|---|---|---|---|---|---|---|---|---|
| Alice Bit   | 0 | 1 | 1 | 0 | 1 | 0 | 0 | 1 | 0 | 1 |
| Alice Basis | Z | X | Z | X | Z | Z | X | X | Z | Z |
| qubit | $\ket{0}$ | $\ket{-}$ | $\ket{1}$ | $\ket{+}$ | $\ket{1}$ | $\ket{0}$ | $\ket{+}$ | $\ket{-}$ | $\ket{0}$ | $\ket{1}$ |

2. Bob measures the qubits in a random basis, eg

| qubit   | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---------|---|---|---|---|---|---|---|---|---|----|
| Alice Basis | Z | X | Z | X | Z | Z | X | X | Z | Z |
| Bob Basis   | Z | Z | X | X | Z | X | X | Z | Z | Z |
| Measurement | 0 | 1 | 0 | 0 | 1 | 1 | 0 | 0 | 0 | 1 |

and his measurement corresponds to his own raw key $(Y_1, Y_2, \dots, Y_n)$.

3. Alice and Bob announce their bases, and Bob tells Alice which ones agree. They don't share their raw keys! They discard the measurement results for bases that disagree.

| qubit   | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---------|---|---|---|---|---|---|---|---|---|----|
| Alice Basis | Z | X | Z | X | Z | Z | X | X | Z | Z |
| Bob Basis   | Z | Z | X | X | Z | X | X | Z | Z | Z |
| Keep        | ✓ | ✗ | ✗ | ✓ | ✓ | ✗ | ✓ | ✗ | ✓ | ✓ |

4. The sifting step: Alice (or Bob) announces *classically* a random sample of the qubits to use to check whether they agree. The sifting theorefore retains the qubits $1, 4, 5, 7, 9, 10$. In this case, the sifted key is, 
$$ 001001 $$
The raw measurement results can disagree because of basis randomness. The sifted key doesn't contain those mismatched-basis results. This way, they know that Eve couldn’t have been measuring many of the qubits.


In [ ]:
n_qubits = 20

alice_bits = random.choices([0, 1], k=n_qubits)
alice_bases = random.choices(['Z', 'X'], k=n_qubits)
bob_bases = random.choices(['Z', 'X'], k=n_qubits)

bob_results = []

print(alice_bits)
print(alice_bases)
print(bob_bases)

In [ ]:
n_qubits = 10

# Alice randomly chooses bits and bases
alice_bits = random.choices([0, 1], k=n_qubits)
alice_bases = random.choices(['Z', 'X'], k=n_qubits)

# Bob randomly chooses measurement bases
bob_bases = random.choices(['Z', 'X'], k=n_qubits)

backend = AerSimulator()

bob_results = []

for i in range(n_qubits):

    qc = QuantumCircuit(1, 1)
    # Alice state preparation
    if alice_bases[i] == 'Z':
        # |0> or |1>
        if alice_bits[i] == 1:
            qc.x(0)

    else:  # X basis
        # |+> or |->
        qc.h(0)

        if alice_bits[i] == 1:
            qc.z(0)

    # Bob measures in his chosen basis
    if bob_bases[i] == 'X':
        qc.h(0)

    qc.measure(0, 0)

    # Run one measurement
    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()
    measured_bit = int(next(iter(counts)))

    bob_results.append(measured_bit)


## Sifting step
sifted_alice = []
sifted_bob = []

for i in range(n_qubits):

    if alice_bases[i] == bob_bases[i]:

        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])


print("Alice bits:   ", alice_bits)
print("Alice bases:  ", alice_bases)
print("Bob bases:    ", bob_bases)
print("Bob results:  ", bob_results)

print()
print("Alice key:    ", sifted_alice)
print("Bob key:      ", sifted_bob)

print()
print("Keys match:", sifted_alice == sifted_bob)

It is also possible for Eve, a third party, to eavesdrop and alter the result. There is a way to extend our BB84 simulation to account for this.

In [ ]:
n_qubits = 100

# Alice randomly chooses bits and bases
alice_bits = random.choices([0, 1], k=n_qubits)
alice_bases = random.choices(['Z', 'X'], k=n_qubits)

# Eve, the eavesdropper, randomly chooses bases
eve_bases = random.choices(['Z', 'X'], k=n_qubits)

# Bob randomly chooses measurement bases
bob_bases = random.choices(['Z', 'X'], k=n_qubits)

backend = AerSimulator()

eve_results = []
bob_results = []

for i in range(n_qubits):

    qc = QuantumCircuit(1, 1)
    # Alice state preparation
    if alice_bases[i] == 'Z':
        # |0> or |1>
        if alice_bits[i] == 1:
            qc.x(0)

    else:  # X basis
        # |+> or |->
        qc.h(0)

        if alice_bits[i] == 1:
            qc.z(0)

    # Eve chooses a basis and measures Alice's qubit
    if eve_bases[i] == 'X':
        qc.h(0)

    qc.measure(0, 0)

    compiled = transpile(qc, backend)
    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()
    eve_result = int(next(iter(counts)))

    eve_results.append(eve_result)

    # Eve sends to Bob
    new_qc = QuantumCircuit(1, 1)
    
    if eve_bases[i] == 'Z':
        # |0> or |1>
        if eve_results[i] == 1:
            new_qc.x(0)

    else:  # X basis
        # |+> or |->
        new_qc.h(0)

        if eve_results[i] == 1:
            new_qc.z(0)
    
    # Bob measures in his chosen basis
    if bob_bases[i] == 'X':
        new_qc.h(0)

    new_qc.measure(0, 0)

    # Run one measurement
    compiled = transpile(new_qc, backend)
    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()
    measured_bit = int(next(iter(counts)))

    bob_results.append(measured_bit)


## Sifting step
sifted_alice = []
sifted_bob = []

for i in range(n_qubits):

    if alice_bases[i] == bob_bases[i]:

        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])


print("Alice bits:   ", alice_bits)
print("Alice bases:  ", alice_bases)
print("Bob bases:    ", bob_bases)
print("Bob results:  ", bob_results)

print()
print("Alice key:    ", sifted_alice)
print("Bob key:      ", sifted_bob)

print()
print("Keys match:", sifted_alice == sifted_bob)


# QBER
errors = sum(
    x != y
    for x, y in zip(sifted_alice, sifted_bob)
)

if len(sifted_alice) > 0:
    qber = errors / len(sifted_alice)
else:
    qber = 0

print()
print("Errors:", errors)
print("Sifted bits:", len(sifted_alice))
print("QBER:", qber)



# Device-independent quantum key distribution (DIQKD)
In the BB84 case, one needs to be sure that the systems and the measurement devices can be trusted. We now discuss situations where this need not be the case, and only through the statistics of these systems we will have information on their securities (for example, through the violation of the Bell inequality). In particular we'll build a protocol where the correlations of entangled particles are used, and a Bell/CHSH test allows Alice and Bob to verify that those correlations are genuinely quantum.

### E91
In the E91 protocol, Charlie shares a singlet state (entangled Bell state) with Alice and Bob. If they use the same basis to measure their respective qubits ($X$ or $Z$), they are guaranteed by the already built correlation of the singlet state that they will get opposite results. If Eve attempts to eavesdrop, this destroys these correlations in a detectable way.

In [ ]:
alice = QuantumRegister(1, "Alice")
bob = QuantumRegister(1, "Bob")

#qr = QuantumRegister(2, name="qr")
cr = ClassicalRegister(2, name="cr")

# Charlie creates the entangled state
# and sends qr0 to Alice and qr1 to Bob
qc = QuantumCircuit(alice, bob, cr)
# Charlie creates the singlet
qc.x(alice[0])
qc.h(alice[0])
qc.cx(alice[0], bob[0])
qc.x(bob[0])

print(qc)

# Measurement #
# Alice and Bob measure in the Z basis


# Alternatively, Alice and Bob measure in the X basis
#qc.h(alice[0])
#qc.h(bob[0])

# If they measure at an arbitrary angle (different from each other)
# all outcomes are equally possible
alice_angle = 0
qc.ry(2*alice_angle, alice[0])

bob_angle = -np.pi/8#math.radians(-45)
qc.ry(2*bob_angle, bob[0])

# Measurement
qc.measure(alice[0], cr[0])
qc.measure(bob[0], cr[1])

backend = AerSimulator()

compiled = transpile(qc, backend)
shots = 10000
result = backend.run(compiled, shots=shots).result()

counts = result.get_counts()
print(counts)

n00 = counts.get('00', 0)
n01 = counts.get('01', 0)
n10 = counts.get('10', 0)
n11 = counts.get('11', 0)

correlation = (n00 + n11 - n01 - n10) / shots

print("E(A1, B1) =", correlation)


In [ ]:
def calculate_correlation(alice_bloch_angle, bob_bloch_angle, shots=10000):
    alice = QuantumRegister(1, "Alice")
    bob = QuantumRegister(1, "Bob")
    cr = ClassicalRegister(2, "cr")

    qc = QuantumCircuit(alice, bob, cr)

    # Prepare singlet state
    qc.x(alice[0])
    qc.h(alice[0])
    qc.cx(alice[0], bob[0])
    qc.x(bob[0])

    # Rotate measurement axes
    qc.ry(2*alice_bloch_angle, alice[0])
    qc.ry(2*bob_bloch_angle, bob[0])

    # Measure
    qc.measure(alice[0], cr[0])
    qc.measure(bob[0], cr[1])

    backend = AerSimulator()
    compiled = transpile(qc, backend)

    result = backend.run(compiled, shots=shots).result()
    counts = result.get_counts()

    n00 = counts.get("00", 0)
    n01 = counts.get("01", 0)
    n10 = counts.get("10", 0)
    n11 = counts.get("11", 0)

    correlation = (n00 + n11 - n01 - n10) / shots

    return correlation, counts

alice_settings = ['Z0', 'Zpi8', 'Zpi4']
bob_settings = ['Z0', 'Zpi8', 'Zminus_pi8']

alice_angles = {
    'Z0': 0,
    'Zpi8': np.pi/8,
    'Zpi4': np.pi/4
}

bob_angles = {
    'Z0': 0,
    'Zpi8': np.pi/8,
    'Zminus_pi8': -np.pi/8
}

alice_choices = []
bob_choices = []

alice_results = []
bob_results = []

# 1. Alice and Bob independently choose random bases
alice_basis = random.choice(["Z0", "Zpi8", "Zpi4"])
bob_basis   = random.choice(["Z0", "Zpi8", "Zminus_pi8"])

def measure_photon(alice_angle, bob_angle):
    alice = QuantumRegister(1, "Alice")
    bob = QuantumRegister(1, "Bob")
    cr = ClassicalRegister(2, "cr")

    qc = QuantumCircuit(alice, bob, cr)

    # Prepare singlet state
    qc.x(alice[0])
    qc.h(alice[0])
    qc.cx(alice[0], bob[0])
    qc.x(bob[0])

    # Rotate measurement axes
    qc.ry(2*alice_angle, alice[0])
    qc.ry(2*bob_angle, bob[0])

    # Measure
    qc.measure(alice[0], cr[0])
    qc.measure(bob[0], cr[1])

    backend = AerSimulator()
    compiled = transpile(qc, backend)

    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()
    bitstring = list(counts.keys())[0]  # e.g. '01'

    # Qiskit little-endian: bitstring[1]=alice, bitstring[0]=bob
    alice_bit = int(bitstring[1])
    bob_bit = int(bitstring[0])
    return alice_bit, bob_bit
    
def run_e91(n_photons):
    records = []
    for _ in range(n_photons):
        a_basis = random.choice(alice_settings)
        b_basis = random.choice(bob_settings)
        a_angle = alice_angles[a_basis]
        b_angle = bob_angles[b_basis]
        a_bit, b_bit = measure_photon(a_angle, b_angle)
        records.append({
            'alice_basis': a_basis,
            'bob_basis': b_basis,
            'alice_bit': a_bit,
            'bob_bit': b_bit
        })
    return pd.DataFrame(records)

'''
E_A0_B0 = calculate_correlation(0, np.pi/8)[0]
E_A0_B1 = calculate_correlation(0, -np.pi/8)[0]
E_A1_B0 = calculate_correlation(np.pi/4, np.pi/8)[0]
E_A1_B1 = calculate_correlation(np.pi/4, -np.pi/8)[0]

S = E_A0_B0 + E_A0_B1 + E_A1_B0 - E_A1_B1

print("E(0, pi/8)   =", E_A0_B0)
print("E(0, -45)  =", E_A0_B1)
print("E(90, 45)  =", E_A1_B0)
print("E(90, -45) =", E_A1_B1)
print("|S| =", abs(S))
'''

In [ ]:
import pandas as pd

alice_settings = ['Z0', 'Zpi8', 'Zpi4']
bob_settings = ['Z0', 'Zpi8', 'Zminus_pi8']

alice_angles = {
    'Z0': 0,
    'Zpi8': np.pi/8,
    'Zpi4': np.pi/4
}

bob_angles = {
    'Z0': 0,
    'Zpi8': np.pi/8,
    'Zminus_pi8': -np.pi/8
}

alice_choices = []
bob_choices = []

alice_results = []
bob_results = []

# 1. Alice and Bob independently choose random bases
alice_basis = random.choice(["Z0", "Zpi8", "Zpi4"])
bob_basis   = random.choice(["Z0", "Zpi8", "Zminus_pi8"])

def measure_photon(alice_angle, bob_angle):
    alice = QuantumRegister(1, "Alice")
    bob = QuantumRegister(1, "Bob")
    cr = ClassicalRegister(2, "cr")

    qc = QuantumCircuit(alice, bob, cr)

    # Prepare singlet state
    qc.x(alice[0])
    qc.h(alice[0])
    qc.cx(alice[0], bob[0])
    qc.x(bob[0])

    # Rotate measurement axes
    qc.ry(2*alice_angle, alice[0])
    qc.ry(2*bob_angle, bob[0])

    # Measure
    qc.measure(alice[0], cr[0])
    qc.measure(bob[0], cr[1])

    backend = AerSimulator()
    compiled = transpile(qc, backend)

    result = backend.run(compiled, shots=1).result()

    counts = result.get_counts()
    bitstring = list(counts.keys())[0]  # e.g. '01'

    # Qiskit little-endian: bitstring[1]=alice, bitstring[0]=bob
    alice_bit = int(bitstring[1])
    bob_bit = int(bitstring[0])
    return alice_bit, bob_bit
    
def run_e91(n_photons):
    records = []
    for _ in range(n_photons):
        a_basis = random.choice(alice_settings)
        b_basis = random.choice(bob_settings)
        a_angle = alice_angles[a_basis]
        b_angle = bob_angles[b_basis]
        a_bit, b_bit = measure_photon(a_angle, b_angle)
        records.append({
            'alice_basis': a_basis,
            'bob_basis': b_basis,
            'alice_bit': a_bit,
            'bob_bit': b_bit
        })
    return pd.DataFrame(records)

def compute_chsh_from_records(records):

    chsh_pairs = [
        ('Z0',   'Zpi8'),
        ('Z0',   'Zminus_pi8'),
        ('Zpi4',  'Zpi8'),
        ('Zpi4',  'Zminus_pi8')
    ]

    correlations = {}

    for a_basis, b_basis in chsh_pairs:

        subset = records[
            (records['alice_basis'] == a_basis) &
            (records['bob_basis'] == b_basis)
        ]

        n = len(subset)

        if n == 0:
            raise ValueError(
                f"No measurements for ({a_basis}, {b_basis})"
            )

        same = (
            subset['alice_bit'] == subset['bob_bit']
        ).sum()

        diff = n - same

        E = (same - diff) / n

        correlations[(a_basis, b_basis)] = E

        print(
            f"E({a_basis}, {b_basis}) = {E:+.4f} "
            f"(N={n})"
        )

    S = (
        correlations[('Z0', 'Zpi8')]
        + correlations[('Z0', 'Zminus_pi8')]
        + correlations[('Zpi4', 'Zpi8')]
        - correlations[('Zpi4', 'Zminus_pi8')]
    )

    return S, correlations

def run_e91_fast(shots_per_pair):
    # Build all 9 circuits
    circuits = []
    labels = []
    
    for a_basis in alice_settings:
        for b_basis in bob_settings:
            a_angle = alice_angles[a_basis]
            b_angle = bob_angles[b_basis]
            
            alice = QuantumRegister(1, "Alice")
            bob = QuantumRegister(1, "Bob")
            cr = ClassicalRegister(2, "cr")
            qc = QuantumCircuit(alice, bob, cr)
            
            # Singlet state
            qc.x(alice[0])
            qc.h(alice[0])
            qc.cx(alice[0], bob[0])
            qc.x(bob[0])
            
            # Measurement bases
            qc.ry(2*a_angle, alice[0])
            qc.ry(2*b_angle, bob[0])
            qc.measure(alice[0], cr[0])
            qc.measure(bob[0], cr[1])
            
            circuits.append(qc)
            labels.append((a_basis, b_basis))
    
    # Transpile all at once
    backend = AerSimulator()
    compiled = transpile(circuits, backend)
    
    # Run all at once
    result = backend.run(compiled, shots=shots_per_pair).result()
    
    # Extract records
    records = []
    for i, (a_basis, b_basis) in enumerate(labels):
        counts = result.get_counts(i)
        for bitstring, count in counts.items():
            alice_bit = int(bitstring[1])
            bob_bit = int(bitstring[0])
            for _ in range(count):
                records.append({
                    'alice_basis': a_basis,
                    'bob_basis': b_basis,
                    'alice_bit': alice_bit,
                    'bob_bit': bob_bit
                })
    
    return pd.DataFrame(records)
#records = run_e91(50)
#display(records.head(10))
    

In [ ]:
# Same basis -> key generation
same_basis = records[records['alice_basis'] == records['bob_basis']]

# Different basis -> CHSH test
#diff_basis = records[records['alice_basis'] != records['bob_basis']]
# All measurements where Alice and Bob used different bases
different_basis = records[
    records['alice_basis'] != records['bob_basis']
]
print(f"Total photons: {len(records)}")
print(f"Same basis (key): {len(same_basis)}")
print(f"Different basis (CHSH): {len(different_basis)}")

alice_key = same_basis['alice_bit'].tolist()
bob_key = [1 - b for b in same_basis['bob_bit'].tolist()]

print(f"Alice key: {alice_key[:10]}")
print(f"Bob key:   {bob_key[:10]}")
print(f"Keys match: {alice_key == bob_key}")

In [ ]:
def compute_chsh_from_records(records):

    chsh_pairs = [
        ('Z0',   'Zpi8'),
        ('Z0',   'Zminus_pi8'),
        ('Zpi4',  'Zpi8'),
        ('Zpi4',  'Zminus_pi8')
    ]

    correlations = {}

    for a_basis, b_basis in chsh_pairs:

        subset = records[
            (records['alice_basis'] == a_basis) &
            (records['bob_basis'] == b_basis)
        ]

        n = len(subset)

        if n == 0:
            raise ValueError(
                f"No measurements for ({a_basis}, {b_basis})"
            )

        same = (
            subset['alice_bit'] == subset['bob_bit']
        ).sum()

        diff = n - same

        E = (same - diff) / n

        correlations[(a_basis, b_basis)] = E
        '''
        print(
            f"E({a_basis}, {b_basis}) = {E:+.4f} "
            f"(N={n})"
        )
        '''
    S = (
        correlations[('Z0', 'Zpi8')]
        + correlations[('Z0', 'Zminus_pi8')]
        + correlations[('Zpi4', 'Zpi8')]
        - correlations[('Zpi4', 'Zminus_pi8')]
    )

    return S, correlations
'''
Nvalues = [1000, 2000, 3000, 4000, 5000, 10000]
Svalues = []

for N in Nvalues:
    print(f"N = {N}")
    records = run_e91(N)

    S, correlations = compute_chsh_from_records(records)
    Svalues.append(S)

print(f"\nS = {S:.4f}")
print(f"|S| = {abs(S):.4f}")
print(f"Classical bound: 2")
print(f"Quantum maximum: {2*np.sqrt(2):.4f}")
print(f"Bell inequality violated: {abs(S) > 2}")
'''    

In [ ]:
import matplotlib.pyplot as plt
Svalues = [np.abs(S) for S in Svalues]

for N, S in zip(Nvalues, Svalues):
    print(f"(N, |S|) = ({N}, {S})")

plt.plot(Nvalues, Svalues, "o-")
plt.hlines(2*np.sqrt(2), 0, Nvalues[-1], linestyles = '--')
plt.xlabel("Nqubits")
plt.ylabel("|S|")
plt.show()

In [ ]:
def analyze_e91(n_photons, n_experiments=50):
    S_values = []
    key_lengths = []
    
    for _ in range(n_experiments):
        records = run_e91_fast(n_photons)
        same = records[records['alice_basis'] == records['bob_basis']]
        diff = records[records['alice_basis'] != records['bob_basis']]
        
        S, _ = compute_chsh_from_records(diff)
        S_values.append(abs(S))
        key_lengths.append(len(same))
    
    return {
        'n_photons': n_photons,
        'mean_S': np.mean(S_values),
        'std_S': np.std(S_values),
        'mean_key_length': np.mean(key_lengths)
    }

results = []
Nvalues = [100, 500, 1000, 2000, 5000]

for N in Nvalues:
    print(f"Nqubits={N}")
    r = analyze_e91(N, n_experiments=50)
    results.append(r)
    print(f"N={N}: <|S|> = {r['mean_S']:.4f} ± {r['std_S']:.4f}, "
          f"key length ≈ {r['mean_key_length']:.0f}")

df = pd.DataFrame(results)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

# <|S|> with error bars
ax.errorbar(
    df['n_photons'], df['mean_S'], yerr=df['std_S'],
    fmt='o-', capsize=5, label=r'$\langle|S|\rangle$', color='steelblue'
)

# Theoretical prediction
ax.axhline(
    y=2*np.sqrt(2), color='red', linestyle='--',
    label=r'Tsirelson bound $2\sqrt{2}$'
)

# Classical bound
ax.axhline(
    y=2, color='gray', linestyle=':',
    label='Classical bound $|S| = 2$'
)

ax.set_xlabel('Number of photons $N$')
ax.set_ylabel(r'$\langle|S|\rangle$')
ax.set_title('E91 CHSH convergence')
ax.legend()
ax.set_ylim(1.5, 3.2)
ax.set_xscale('log')
plt.tight_layout()
#plt.savefig('e91_convergence.png', dpi=150)
plt.show()